In [ ]:
# This script is the same as rnn-teacher-forcing.ipynb except for the fact, that it embeds the label for each real sequence in a vector which is fed as an additional input.

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle as pkl
import random
from model import SequenceGeneratorEmbedded


# DATA LOADING
DATASETS_PATH = os.path.join('..', '..', '..','data')
TEST_DATASET_PATH = os.path.join(DATASETS_PATH, 'test.pickle')
TRAIN_DATASET_PATH = os.path.join(DATASETS_PATH, 'train.pickle')

with open(TEST_DATASET_PATH, 'rb') as f:
    test_dataset = pkl.load(f)
    print(test_dataset['label'].value_counts())

with open(TRAIN_DATASET_PATH, 'rb') as f:
    train_dataset = pkl.load(f)
    print(train_dataset['label'].value_counts())

# DATASET CLASS
class SensorDataSet(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __getitem__(self, idx):
        data = self.dataset.iloc[idx]
        x = torch.tensor(data['sensor_data'], dtype=torch.float32)
        y = torch.tensor(data['label'], dtype=torch.long)
        return x, y

    def __len__(self):
        return len(self.dataset)



# TRAINING FUNCTION
def train_generator(generator, train_loader, num_epochs=10, lr=0.001, teacher_forcing_start=1.0, teacher_forcing_end=0.0):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    generator.to(device)
    optimizer = torch.optim.Adam(generator.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    generator.train()
    total_steps = len(train_loader) * num_epochs
    step = 0

    for epoch in range(num_epochs):
        total_loss = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device) # (batch, 128, 6), (batch,)
            batch_size, seq_len, input_dim = x.size()

            # Input for first timestep
            input_t = x[:, 0, :].unsqueeze(1) # first timestep: (batch, 1, 6)

            # Scheduled teacher forcing rate: how much real data gets fed for this timestep?
            teacher_forcing_ratio = teacher_forcing_end + (teacher_forcing_start - teacher_forcing_end) * np.exp(-5 * step / total_steps)
            step += 1

            hidden = None
            outputs = []

            # Iterate through sequence timestep by timestep and feed real values x or generated values output as the next input according to the current teacher forcing rate
            for t in range(1, seq_len):
                output, hidden = generator(input_t, y, hidden) # output: (batch, 1, 6)
                outputs.append(output)

                use_teacher = torch.rand(batch_size, device=device) < teacher_forcing_ratio # determine who out of the batch gets real and who gets generated data
                use_teacher = use_teacher.unsqueeze(1).unsqueeze(2) # (batch, 1, 1)

                next_input = use_teacher * x[:, t, :].unsqueeze(1) + (~use_teacher) * output.detach()
                input_t = next_input

            # Output and target for whole sequence
            outputs = torch.cat(outputs, dim=1) # (batch, seq_len - 1, 6)
            target = x[:, 1:, :] # (batch, seq_len - 1, 6)

            # Loss between output and target
            loss = loss_fn(outputs, target)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * x.size(0)

        print(f"[Epoch {epoch+1}] Loss: {total_loss / len(train_loader.dataset):.6f}")

# INITIALIZING AND TRAINING
input_dim = 6 # amount of sensors
label_embed_dim = 10 # size of the vector where the labels are imbedded
hidden_dim = 128
num_layers = 2
num_labels = 12

generator = SequenceGeneratorEmbedded(input_dim, label_embed_dim, hidden_dim, num_layers, num_labels)

train_loader = DataLoader(SensorDataSet(train_dataset), batch_size=32, shuffle=True)

train_generator(generator, train_loader, num_epochs=10, lr=0.001)

In [ ]:
# Export generator model
model_dir = os.path.join('..', 'models')
os.makedirs(model_dir, exist_ok=True)

torch.save(generator.state_dict(), os.path.join(model_dir,'generator-rnn.pt') )

In [ ]:
# SEQUENCE GENERATION FUNCTION
def generate_sequence(generator, seed, label, seq_len=128):
    generator.eval()
    device = next(generator.parameters()).device

    # Label we want to generate for
    label = torch.tensor([label], dtype=torch.long).to(device)

    # Format starting seed as (1, 1, 6) tensor, because the LSTM wants those dimensions
    seed = torch.tensor(seed, dtype=torch.float32).unsqueeze(0).unsqueeze(1).to(device)  # (1, 1, sensor_dim)

    generated = [seed.squeeze(0)] # Store first sequence step
    hidden = None

    # Regressively generate a sequence by using last output as next input
    for _ in range(seq_len - 1):
        output, hidden = generator(seed, label, hidden)
        seed = output
        generated.append(output.squeeze(0))

    return torch.cat(generated, dim=0).cpu().detach() # generated sequence: (128, 6)

# PLOTTING FUNCTION
def plot_sequence(dataset, return_figure=True):
    fig, axes = plt.subplots(3,2)

    directions = ["x", "y", "z"]

    value_type = ['acceleration', 'rotation']

    for i, direction_label in enumerate(directions):
        for j, value_label in enumerate(value_type):
            axes[i,j].plot(dataset[...,:,3*j+i])
            if i == 0:
                axes[i,j].set_title(value_label)
                
            if j == 1:
                axes[i,j].text(1, 0.5,direction_label, size=12, rotation=270, transform=axes[i,j].transAxes)
    if return_figure:
        return fig

# PLOT A RANDOM REAL AND A GENERATED SEQUENCE PER LABEL
label_targets = [0, 1, 2]  # Labels we want to plot

# Collect mean and std of real starting values for generating a new random starting value
start_values = np.stack(train_dataset['sensor_data'].apply(lambda x: x[0]))  # shape: (num_samples, 6)
mean_start = start_values.mean(axis=0)
std_start = start_values.std(axis=0)

plotted_labels = set()

for batch_x, batch_y in train_loader:
    for label in label_targets:
        if label in plotted_labels:
            continue  # Already plotted

        # Find a real example for this label
        indices = (batch_y == label).nonzero(as_tuple=True)[0]
        if len(indices) > 0:
            idx = random.choice(indices).item()
            real_example = batch_x[idx].numpy()  # (128, 6)

            # Generate synthetic sequence
            seed = mean_start + np.random.normal(0, std_start * 0.5, size=mean_start.shape)
            fake_seq = generate_sequence(generator, seed, label)

            # Print and plot both
            print(f"=== Label {label} ===")
            print("Real sequence:")
            plot_sequence(real_example)
            print("Generated sequence:")
            plot_sequence(fake_seq.numpy())

            plotted_labels.add(label)

        if len(plotted_labels) == len(label_targets):
            break

    if len(plotted_labels) == len(label_targets):
        break


In [ ]:
import pandas as pd

def generate_synthetic_dataset(generator, label_counts, seq_len=128):
    device = next(generator.parameters()).device
    sensor_dim = generator.fc.out_features

    # Start values stats from training data for seeding generation
    start_values = np.stack(train_dataset['sensor_data'].apply(lambda x: x[0]))  # shape: (num_samples, sensor_dim)
    mean_start = start_values.mean(axis=0)
    std_start = start_values.std(axis=0)

    all_sequences = []
    all_labels = []

    # For each label ...
    for label in range(len(label_counts)):
        n_samples = label_counts[label]
        # ... generate the desired amount of sequences
        for _ in range(n_samples):
            # Generate a seed around the mean start with some noise
            seed = mean_start + np.random.normal(0, std_start * 0.5, size=mean_start.shape)
            # Generate sequence
            synthetic_seq = generate_sequence(generator, seed, label, seq_len=seq_len)
            synthetic_seq = synthetic_seq.numpy()
            all_sequences.append(synthetic_seq)
            all_labels.append(label)

    df_generated = pd.DataFrame({'sensor_data': all_sequences, 'label': all_labels})
    return df_generated

# Generate datset
label_counts = [100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100]  # Specify how many sequences to generate per label
generated_dataset = generate_synthetic_dataset(generator, label_counts, seq_len=128)

# Save to pickle
GENERATED_DATASET_PATH = os.path.join(DATASETS_PATH, 'generated.pickle')
with open(GENERATED_DATASET_PATH, 'wb') as f:
    pkl.dump(generated_dataset, f)

print(f"Saved generated dataset with {len(generated_dataset)} samples to '{GENERATED_DATASET_PATH}'")


In [ ]:
def evaluate_generated_sequences(train_dataset, generated_dataset):
    label_count = 12

    # Step 1: calculate mean sequence per label
    label_means = {}

    for label in range(label_count):
        sequences = train_dataset[train_dataset['label'] == label]['sensor_data'].to_list()
        if len(sequences) == 0:
            continue
        stacked = np.stack(sequences)  # (num_samples, seq_len, sensor_dim)
        label_means[label] = stacked.mean(axis=0)  # (seq_len, sensor_dim)

    # Step 2: compare generated sequence to mean sequence
    per_label_errors = {label: [] for label in range(label_count)}

    for _, row in generated_dataset.iterrows():
        label = row['label']
        if label not in label_means:
            continue  # skip if no real data for this label
        real_mean_seq = label_means[label]  # (128, 6)
        gen_seq = row['sensor_data']       # (128, 6)

        squared_diff = (gen_seq - real_mean_seq) ** 2  # (128, 6)
        mse_over_time = squared_diff.mean(axis=0)      # (6,)
        total_sensor_mse = mse_over_time.sum()         # skalar
        per_label_errors[label].append(total_sensor_mse)

    # Step 3: calculate loss per label
    average_error_per_label = {}
    for label in range(label_count):
        errors = per_label_errors[label]
        if len(errors) > 0:
            average_error_per_label[label] = np.mean(errors)
        else:
            average_error_per_label[label] = None # no data

    return average_error_per_label


In [ ]:
# Load generated.pickle
with open(os.path.join(DATASETS_PATH, 'generated.pickle'), 'rb') as f:
    generated_dataset = pkl.load(f)

# Evaluate the generated data
error_stats = evaluate_generated_sequences(train_dataset, generated_dataset)

# Results
print("Average loss per label:")
for label in sorted(error_stats):
    val = error_stats[label]
    if val is not None:
        print(f"Label {label}: {val:.4f}")
    else:
        print(f"Label {label}: no data in training set")